# AADS-ULoRA Colab Bootstrap

**Essential entry point for Google Colab deployment**

This notebook sets up the entire AADS-ULoRA environment on Google Colab, including GPU detection, Google Drive mounting, dependency installation, and configuration generation.

## What this notebook does:
1. **GPU Detection & Memory Analysis** - Detects GPU type (A100/T4/P100) and calculates optimal batch sizes
2. **Google Drive Mounting** - Mounts with retry logic (3 attempts) and creates directory structure
3. **Environment Setup** - Installs dependencies and configures Colab-specific settings
4. **Configuration Generation** - Creates `config/colab.json` optimized for your GPU
5. **Validation & Testing** - Verifies installation and runs smoke tests
6. **Next Steps** - Provides clear instructions for continuing the training pipeline

---
**Expected Runtime:** 5-15 minutes (depending on network and GPU)
**Required:** Google Drive with at least 20GB free space

---

**Note:** Run all cells in order. This notebook should be executed BEFORE any other training notebooks.

## 1. GPU Detection & Memory Analysis

First, let's detect what GPU is available and analyze its memory capacity to optimize batch sizes.

In [ ]:
import torch
import psutil
import json
from pathlib import Path
import logging
from datetime import datetime

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def detect_gpu():
    """Detect GPU type and memory capacity."""
    if not torch.cuda.is_available():
        logger.warning("❌ No GPU detected! Colab requires a GPU for training.")
        logger.info("💡 Go to Runtime → Change runtime type → Select 'GPU'")
        return {
            'gpu_type': 'none',
            'total_memory_gb': 0,
            'recommended_batch_size': 2,
            'max_batch_size': 4,
            'warning': 'No GPU available'
        }
    
    # Get GPU properties
    device = torch.device('cuda:0')
    props = torch.cuda.get_device_properties(device)
    name = props.name
    total_memory = props.total_memory / (1024**3)  # Convert to GB
    
    # Detect GPU type from name
    gpu_type = 'unknown'
    if 'A100' in name:
        gpu_type = 'A100'
    elif 'T4' in name:
        gpu_type = 'T4'
    elif 'P100' in name:
        gpu_type = 'P100'
    elif 'V100' in name:
        gpu_type = 'V100'
    elif 'K80' in name:
        gpu_type = 'K80'
    
    # Calculate optimal batch sizes based on memory
    # These are conservative estimates for stable training
    if total_memory >= 40:  # A100 40GB+
        recommended_batch = 32
        max_batch = 64
        gradient_accumulation = 1
    elif total_memory >= 24:  # A100 24GB or V100 32GB
        recommended_batch = 16
        max_batch = 32
        gradient_accumulation = 2
    elif total_memory >= 16:  # T4 16GB, V100 16GB
        recommended_batch = 8
        max_batch = 16
        gradient_accumulation = 2
    elif total_memory >= 12:  # P100 16GB (usually less)
        recommended_batch = 4
        max_batch = 8
        gradient_accumulation = 4
    else:  # K80, low memory
        recommended_batch = 2
        max_batch = 4
        gradient_accumulation = 8
    
    # Check if memory is critically low
    warning = None
    if total_memory < 10:
        warning = "⚠️  Low memory GPU detected (<10GB). Training may be slow or unstable."
        logger.warning(warning)
    elif total_memory < 8:
        warning = "⚠️  Very low memory GPU (<8GB). Consider using a different runtime."
        logger.error(warning)
    
    result = {
        'gpu_type': gpu_type,
        'gpu_name': name,
        'total_memory_gb': round(total_memory, 2),
        'recommended_batch_size': recommended_batch,
        'max_batch_size': max_batch,
        'gradient_accumulation_steps': gradient_accumulation,
        'cuda_version': torch.version.cuda,
        'pytorch_version': torch.__version__,
        'warning': warning
    }
    
    logger.info(f"✅ GPU Detected: {name}")
    logger.info(f"   Type: {gpu_type}")
    logger.info(f"   Memory: {total_memory:.2f} GB")
    logger.info(f"   Recommended batch size: {recommended_batch}")
    logger.info(f"   Gradient accumulation steps: {gradient_accumulation}")
    
    return result

# Detect GPU
gpu_info = detect_gpu()

# Display results
print("\n" + "="*60)
print("GPU DETECTION RESULTS")
print("="*60)
print(f"GPU Type: {gpu_info['gpu_type']}")
print(f"GPU Name: {gpu_info['gpu_name']}")
print(f"Total Memory: {gpu_info['total_memory_gb']} GB")
print(f"Recommended Batch Size: {gpu_info['recommended_batch_size']}")
print(f"Max Batch Size: {gpu_info['max_batch_size']}")
print(f"Gradient Accumulation Steps: {gpu_info['gradient_accumulation_steps']}")
print(f"PyTorch Version: {gpu_info['pytorch_version']}")
print(f"CUDA Version: {gpu_info['cuda_version']}")
if gpu_info['warning']:
    print(f"\n⚠️  WARNING: {gpu_info['warning']}")
print("="*60 + "\n")

## 2. Mount Google Drive

Mount Google Drive with retry logic to persist data across Colab sessions.

In [ ]:
from google.colab import drive
import os
import time
from pathlib import Path

def mount_google_drive_with_retry(max_attempts: int = 3) -> Path:
    """
    Mount Google Drive with retry logic.
    
    Args:
        max_attempts: Number of retry attempts
        
    Returns:
        Path to mounted drive
    """
    mount_point = Path('/content/drive')
    
    for attempt in range(max_attempts):
        try:
            logger.info(f"🔗 Mounting Google Drive (attempt {attempt + 1}/{max_attempts})...")
            
            # Check if already mounted
            if mount_point.exists() and (mount_point / 'MyDrive').exists():
                logger.info("✅ Google Drive already mounted")
            else:
                drive.mount(str(mount_point))
                logger.info("✅ Google Drive mounted successfully")
            
            # Verify mount
            mydrive = mount_point / 'MyDrive'
            if not mydrive.exists():
                raise RuntimeError("MyDrive directory not found after mounting")
            
            return mydrive
            
        except Exception as e:
            logger.error(f"❌ Mount attempt {attempt + 1} failed: {e}")
            if attempt < max_attempts - 1:
                logger.info(f"⏳ Retrying in 3 seconds...")
                time.sleep(3)
            else:
                logger.error("❌ All mount attempts failed. Please check your Google Drive permissions.")
                raise

# Mount Google Drive
try:
    mydrive = mount_google_drive_with_retry(max_attempts=3)
    print(f"\n✅ Google Drive mounted at: {mydrive}")
except Exception as e:
    print(f"\n❌ Failed to mount Google Drive: {e}")
    print("💡 Troubleshooting:")
    print("   - Ensure you're signed into Google Colab with the correct account")
    print("   - Check that you granted drive permissions")
    print("   - Try Runtime → Factory reset runtime and re-run")
    raise

# Create workspace directory
workspace_dir = Path('/content/aads_ulora')
workspace_dir.mkdir(parents=True, exist_ok=True)
logger.info(f"📁 Workspace directory: {workspace_dir}")
print(f"📁 Workspace directory created: {workspace_dir}")

## 3. Environment Setup

Install dependencies and configure Colab-specific settings.

In [ ]:
import subprocess
import sys
from pathlib import Path
import logging

def install_dependencies(requirements_path: Path, max_retries: int = 3):
    """
    Install Python dependencies with retry logic.
    
    Args:
        requirements_path: Path to requirements.txt
        max_retries: Number of retry attempts for network operations
    """
    if not requirements_path.exists():
        logger.error(f"❌ Requirements file not found: {requirements_path}")
        raise FileNotFoundError(f"Requirements file not found: {requirements_path}")
    
    logger.info(f"📦 Installing dependencies from {requirements_path}...")
    
    for attempt in range(max_retries):
        try:
            # Upgrade pip first
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', '-q'
            ])
            
            # Install requirements
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path), '-q'
            ])
            
            logger.info("✅ Dependencies installed successfully")
            return
            
        except subprocess.CalledProcessError as e:
            logger.error(f"❌ Installation attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                logger.info("⏳ Retrying in 5 seconds...")
                time.sleep(5)
            else:
                logger.error("❌ All installation attempts failed")
                raise

def configure_colab_settings():
    """Configure Colab-specific settings."""
    logger.info("⚙️  Configuring Colab settings...")
    
    # Set environment variables
    os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # Async CUDA launches
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF warnings
    
    # Configure PyTorch for optimal performance
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    logger.info("✅ Colab settings configured")

# Add workspace to Python path
sys.path.insert(0, str(workspace_dir))
logger.info(f"📚 Workspace added to Python path: {workspace_dir}")

# Install dependencies
requirements_path = workspace_dir / 'colab_notebooks' / 'requirements_colab.txt'
try:
    install_dependencies(requirements_path)
    print("\n✅ Dependencies installed successfully")
except Exception as e:
    print(f"\n❌ Failed to install dependencies: {e}")
    print("💡 Try running: !pip install -r colab_notebooks/requirements_colab.txt")
    raise

# Configure Colab settings
configure_colab_settings()

print("\n✅ Environment setup complete!")

## 4. Configuration Generation

Generate optimized `config/colab.json` based on detected GPU and system capabilities.

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import copy

def generate_colab_config(gpu_info: dict, config_dir: Path) -> dict:
    """
    Generate Colab configuration based on GPU capabilities.
    
    Args:
        gpu_info: GPU detection results
        config_dir: Path to config directory
        
    Returns:
        Generated configuration dictionary
    """
    logger.info("📝 Generating Colab configuration...")
    
    # Load base configuration if exists
    base_config_path = config_dir / 'base.json'
    if base_config_path.exists():
        with open(base_config_path, 'r') as f:
            config = json.load(f)
        logger.info(f"📋 Loaded base configuration from {base_config_path}")
    else:
        logger.warning("⚠️  base.json not found, creating minimal config")
        config = {
            "api": {
                "host": "0.0.0.0",
                "port": 8000,
                "reload": False,
                "workers": 2
            },
            "database": {
                "type": "sqlite",
                "path": "./data/colab.db"
            },
            "training": {}
        }

    # Ensure colab section exists
    if 'colab' not in config:
        config['colab'] = {}
    
    # Update Colab-specific settings based on GPU
    config['colab'].update({
        'enabled': True,
        'environment': 'google_colab',
        'gpu_type': gpu_info['gpu_type'],
        'drive_mount_path': '/content/drive/MyDrive/aads_ulora',
        'workspace_path': '/content/aads_ulora',
        'cache_dir': '/content/cache'
    })
    
    # Memory optimization settings
    config['colab']['memory_optimization'] = {
        'gradient_checkpointing': True,
        'mixed_precision': True,
        'memory_efficient_attention': True,
        'clear_cache_frequency': 10,
        'max_batch_size_gb4': 8,
        'max_batch_size_gb8': 16,
        'max_batch_size_gb16': 32,
        'max_batch_size_gb24': 64,
        'max_batch_size_gb32': 128
    }
    
    # Training settings optimized for detected GPU
    config['colab']['training'] = {
        'gradient_accumulation_steps': gpu_info['gradient_accumulation_steps'],
        'use_amp': True,
        'pin_memory': True,
        'num_workers': 2,
        'prefetch_factor': 2,
        'persistent_workers': False,
        'checkpoint_interval': 5,
        'early_stopping_patience': 10
    }
    
    # Paths (relative to workspace)
    config['colab']['paths'] = {
        'data_dir': './data',
        'models_dir': './models',
        'checkpoints_dir': './checkpoints',
        'logs_dir': './logs',
        'outputs_dir': './outputs'
    }
    
    # Monitoring settings
    config['colab']['monitoring'] = {
        'enabled': True,
        'track_gpu_memory': True,
        'track_disk_usage': True,
        'progress_bar': True,
        'log_interval': 10,
        'tensorboard': False
    }

    # Update training phase configurations based on GPU memory
    total_memory = gpu_info['total_memory_gb']
    
    # Phase 1 batch size
    if 'training' not in config:
        config['training'] = {}
    if 'stage1' not in config['training']:
        config['training']['stage1'] = {}
    config['training']['stage1'].update({
        'batch_size': min(gpu_info['recommended_batch_size'], 32),
        'gradient_accumulation_steps': gpu_info['gradient_accumulation_steps'],
        'num_epochs': 10,
        'learning_rate': 1e-4,
        'lora_r': 32,
        'lora_alpha': 32,
        'lora_dropout': 0.1,
        'loraplus_lr_ratio': 16
    })
    
    # Phase 2 batch size (Stable Diffusion, more memory intensive)
    if 'phase2' not in config['training']:
        config['training']['phase2'] = {}
    config['training']['phase2'].update({
        'batch_size': max(1, gpu_info['recommended_batch_size'] // 2),
        'gradient_accumulation_steps': gpu_info['gradient_accumulation_steps'] * 2,
        'num_epochs': 5,
        'learning_rate': 1e-4,
        'lora_r': 16,
        'lora_alpha': 16
    })
    
    # Phase 3 batch size
    if 'stage3' not in config['training']:
        config['training']['stage3'] = {}
    config['training']['stage3'].update({
        'batch_size': min(gpu_info['recommended_batch_size'], 16),
        'gradient_accumulation_steps': gpu_info['gradient_accumulation_steps'],
        'num_epochs': 10,
        'learning_rate': 5e-5,
        'lora_r': 8,
        'lora_alpha': 16
    })

    # Add metadata
    config['_metadata'] = {
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'generator': 'colab_bootstrap.ipynb',
        'gpu_info': gpu_info
    }

    # Save configuration
    config_dir.mkdir(parents=True, exist_ok=True)
    output_path = config_dir / 'colab.json'
    
    with open(output_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    logger.info(f"✅ Configuration saved to: {output_path}")
    print(f"\n✅ Configuration generated and saved to: {output_path}")
    
    return config

# Generate configuration
config_dir = workspace_dir / 'config'
colab_config = generate_colab_config(gpu_info, config_dir)

# Display summary
print("\n" + "="*60)
print("CONFIGURATION SUMMARY")
print("="*60)
print(f"GPU Type: {colab_config['colab']['gpu_type']}")
print(f"Batch Size (Stage 1): {colab_config['training']['stage1']['batch_size']}")
print(f"Batch Size (Phase 2): {colab_config['training']['phase2']['batch_size']}")
print(f"Batch Size (Stage 3): {colab_config['training']['stage3']['batch_size']}")
print(f"Gradient Accumulation: {colab_config['colab']['training']['gradient_accumulation_steps']}")
print(f"Mixed Precision: {colab_config['colab']['training']['use_amp']}")
print(f"Config Location: {config_dir / 'colab.json'}")
print("="*60 + "\n")

## 5. Validation & Smoke Tests

Verify the installation and configuration are working correctly.

In [ ]:
import importlib
import sys
from pathlib import Path
import json

def validate_installation() -> bool:
    """
    Validate that all required modules can be imported and configuration is valid.
    
    Returns:
        True if validation passes
    """
    logger.info("🔍 Running validation checks...")
    all_passed = True
    
    # Check Python path
    print("\n📋 Validation Checks:")
    print("-" * 40)
    
    # 1. Check workspace in path
    if str(workspace_dir) in sys.path:
        print("✅ Workspace in Python path")
    else:
        print("❌ Workspace not in Python path")
        all_passed = False
    
    # 2. Check config file exists
    config_path = workspace_dir / 'config' / 'colab.json'
    if config_path.exists():
        print(f"✅ Configuration file exists: {config_path}")
        try:
            with open(config_path, 'r') as f:
                config = json.load(f)
            print(f"   - GPU Type: {config['colab']['gpu_type']}")
            print(f"   - Batch Size (S1): {config['training']['stage1']['batch_size']}")
        except Exception as e:
            print(f"❌ Configuration file invalid: {e}")
            all_passed = False
    else:
        print(f"❌ Configuration file missing: {config_path}")
        all_passed = False
    
    # 3. Check required directories
    required_dirs = ['data', 'models', 'checkpoints', 'logs', 'outputs']
    for dir_name in required_dirs:
        dir_path = workspace_dir / dir_name
        if dir_path.exists():
            print(f"✅ Directory exists: {dir_name}")
        else:
            print(f"⚠️  Directory will be created: {dir_name}")
            dir_path.mkdir(parents=True, exist_ok=True)
    
    # 4. Check critical imports
    critical_imports = [
        ('torch', 'PyTorch'),
        ('transformers', 'HuggingFace Transformers'),
        ('peft', 'PEFT'),
        ('accelerate', 'Accelerate'),
        ('numpy', 'NumPy'),
        ('pandas', 'Pandas'),
        ('PIL', 'Pillow'),
        ('sklearn', 'Scikit-learn'),
        ('fastapi', 'FastAPI'),
        ('pydantic', 'Pydantic')
    ]
    
    print("\n📦 Dependency Checks:")
    print("-" * 40)
    for module, name in critical_imports:
        try:
            importlib.import_module(module)
            print(f"✅ {name}")
        except ImportError as e:
            print(f"❌ {name} - Not installed: {e}")
            all_passed = False
    
    # 5. Check CUDA
    print("\n🎮 GPU Status:")
    print("-" * 40)
    if torch.cuda.is_available():
        print(f"✅ CUDA Available: {torch.version.cuda}")
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
        memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"✅ GPU Memory: {memory_gb:.2f} GB")
        
        # Test CUDA tensor operation
        try:
            x = torch.randn(10, 10).cuda()
            y = x * 2
            print("✅ CUDA operations working")
        except Exception as e:
            print(f"❌ CUDA operations failed: {e}")
            all_passed = False
    else:
        print("❌ CUDA not available")
        all_passed = False
    
    # 6. Check disk space
    print("\n💾 Disk Space:")
    print("-" * 40)
    try:
        import shutil
        total, used, free = shutil.disk_usage('/content')
        free_gb = free / (1024**3)
        print(f"Total: {total / (1024**3):.1f} GB")
        print(f"Used: {used / (1024**3):.1f} GB")
        print(f"Free: {free_gb:.1f} GB")
        
        if free_gb < 10:
            print("⚠️  Low disk space (<10GB free). Consider freeing up space.")
        else:
            print("✅ Sufficient disk space")
    except Exception as e:
        print(f"⚠️  Could not check disk space: {e}")
    
    print("\n" + "="*60)
    if all_passed:
        print("✅ ALL VALIDATION CHECKS PASSED")
    else:
        print("❌ SOME VALIDATION CHECKS FAILED")
        print("Please review errors above and fix before proceeding.")
    print("="*60 + "\n")
    
    return all_passed

# Run validation
validation_passed = validate_installation()

if not validation_passed:
    logger.error("Validation failed! Please fix issues before proceeding.")
    raise RuntimeError("Installation validation failed")

## 6. Next Steps

The environment is now ready! Here's what to do next:

### ✅ Setup Complete!

Your Colab environment is now configured with:
- ✅ GPU detection and optimization
- ✅ Google Drive mounted at `/content/drive/MyDrive/aads_ulora`
- ✅ Dependencies installed
- ✅ Configuration generated at `config/colab.json`
- ✅ Validation passed

### 📚 Recommended Next Steps:

1. **Data Preparation** (Notebook 1)
   - Run `1_data_preparation.ipynb` to download and preprocess datasets
   - This will create train/validation/test splits
   - Expected time: 10-30 minutes

2. **Continual Adapter Training** (Notebook 2)
   - Run `2_continual_sd_lora_training.ipynb` for continual adapter training
   - Trains on DINOv3-giant with your crop data
   - Expected time: 2-4 hours

3. **Phase 2 Training** (Notebook 3)
   - Run `3_phase2_training.ipynb` for Stable Diffusion LoRA
   - Generates synthetic disease images
   - Expected time: 1-2 hours

4. **Final Adapter Refinement** (Notebook 4)
   - Run `4_final_refinement_training.ipynb` for final adapter refinement
   - Final model with OOD detection
   - Expected time: 2-3 hours

5. **Testing & Validation** (Notebook 5)
   - Run `5_testing_validation.ipynb` to evaluate model performance
   - Generate metrics and visualizations
   - Expected time: 30 minutes

6. **Performance Monitoring** (Notebook 6)
   - Run `6_performance_monitoring.ipynb` for ongoing monitoring
   - Set up dashboards and alerts
   - Expected time: 15 minutes

### 💡 Tips:

- **Save checkpoints regularly** - They're saved to Google Drive automatically
- **Monitor GPU memory** - Use the built-in monitoring to avoid OOM errors
- **Check logs** - All logs are saved to `./logs/` directory
- **Restart runtime if needed** - If you encounter memory issues, restart and re-run this bootstrap notebook

### 🆘 Troubleshooting:

**Issue:** Out of memory (OOM) errors
**Solution:** Reduce batch size in `config/colab.json` and increase gradient accumulation steps

**Issue:** Slow training
**Solution:** Check GPU type - T4 is slower than A100. Consider upgrading runtime.

**Issue:** Import errors
**Solution:** Re-run this notebook from the beginning to ensure all dependencies are installed

**Issue:** Google Drive disconnects
**Solution:** Re-mount drive and continue from the last checkpoint

---

**Ready to start?** Open `1_data_preparation.ipynb` to begin the training pipeline!


In [ ]:
# Final verification - print workspace structure
print("\n📁 Workspace Structure:")
print("="*60)

workspace = Path('/content/aads_ulora')
if workspace.exists():
    for item in sorted(workspace.iterdir()):
        if item.is_dir():
            print(f"📁 {item.name}/")
        else:
            size_mb = item.stat().st_size / (1024*1024)
            print(f"📄 {item.name} ({size_mb:.1f} MB)")
else:
    print("⚠️  Workspace not found")

print("="*60)
print("\n🎉 Bootstrap complete! You're ready to train.")
print("👉 Next: Run 1_data_preparation.ipynb")